In [1]:
!pip install -U langchain langchain-community langchain-core langchain-groq faiss-cpu sentence-transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.9/112.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Fou

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
folder_path = "/content/drive/MyDrive/RAG"

In [7]:
import os

print(os.listdir(folder_path))

['UGE_FY_Syllabus_AY_2020-21.pdf', 'SY CSE Structure and Syllabus V13.pdf', 'Revised SY CSE Structure and Syllabus 2021-22.docx.pdf', 'Prerevised SY Syllabus 2017.docx.pdf', 'Pre-revised TY Syllabus 2018.doc.pdf', 'Final Year CSE Syllabus 2023-24 ver-12.pdf', 'Final_Year_B.Tech._CSE_Syllabus_v2.pdf', 'TY_CSE Structure and Syllabus V9.pdf']


In [8]:
from langchain_community.document_loaders import PyPDFLoader

all_docs = []

for file in os.listdir(folder_path):
    if file.endswith(".pdf"):
        path = os.path.join(folder_path, file)

        loader = PyPDFLoader(path)
        docs = loader.load()

        for doc in docs:
            doc.metadata["source"] = file

        all_docs.extend(docs)

print("Loaded pages:", len(all_docs))

Loaded pages: 349


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_docs)

print("Total chunks:", len(chunks))

Total chunks: 1964


In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_8124/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [13]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key="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",
    model="llama-3.3-70b-versatile"
)

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
Answer ONLY from the context below.

Context:
{context}

Question:
{question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

In [15]:
def ask(query):
    docs = retriever.get_relevant_documents(query)

    print("\n📄 Sources:")
    for d in docs:
        print("-", d.metadata.get("source"))

    context = format_docs(docs)
    response = llm.invoke(prompt.format(context=context, question=query))

    print("\n🤖 Answer:\n", response.content)

In [16]:
vectorstore.save_local("syllabus_db")

In [17]:
while True:
    query = input("\nAsk your question (type 'exit' to quit): ")

    if query.lower() == "exit":
        break

    response = rag_chain.invoke(query)

    print("\n🤖 Answer:\n", response.content)


Ask your question (type 'exit' to quit): What subjects are in second year CSE?

🤖 Answer:
 The context provided does not explicitly mention the subjects for the second year of CSE. However, it lists various subjects under different electives and core subjects for Semester Examination-I (SE-I) and Semester Examination-II (SE-II), which may be part of the second year curriculum, but the year is not specified.

Some of the subjects mentioned are:
- CSL403 Image Processing
- CSL406 Cloud Computing
- CSL404 Human Computer Interaction
- CSL407 Internet of Things
- CSP411 Cloud Computing Lab
- CSP412 Internet of Things Lab 

And also:
- CSL417 Agile Technology
- CSL418 Software Testing and Quality Assurance
- CSL420 Cyber Security
- CSP425 Cyber Security Lab 

These subjects seem to be part of the professional subjects - core and electives, but the exact year is not confirmed.

Ask your question (type 'exit' to quit): What is the syllabus of Data Structures?

🤖 Answer:
 The syllabus of Data 